In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'diffusers', 'peft', 'transformers', 'accelerate', 'scipy'], check=True)
print('Done. Restart kernel: Session → Restart & Run All')


In [ ]:
import gc
import os
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

SEED = 13
random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if hasattr(torch, 'mps'):
    torch.mps.manual_seed(SEED)

IS_KAGGLE = os.path.exists('/kaggle/working')

if torch.cuda.is_available():
    DEVICE = 'cuda'
elif hasattr(torch, 'mps') and torch.backends.mps.is_available():
    DEVICE = 'mps'
else:
    DEVICE = 'cpu'
print('device:', DEVICE, '| kaggle:', IS_KAGGLE)

In [ ]:
import torch
from torch.utils.data import Dataset
from torchvision import transforms
from PIL import Image
from diffusers import StableDiffusionPipeline, DDPMScheduler, AutoencoderKL, UNet2DConditionModel
from transformers import CLIPTextModel, CLIPTokenizer, CLIPModel, CLIPProcessor
from peft import LoraConfig
from tqdm.auto import tqdm

SD_MODEL = 'stable-diffusion-v1-5/stable-diffusion-v1-5'
WEIGHTS_DIR = Path('/kaggle/working/lora_out') if IS_KAGGLE else Path('./lora_out')
RESULTS_DIR = Path('/kaggle/working/generated') if IS_KAGGLE else Path('./generated')
PHOTOS_DIR = Path('/kaggle/input/datasets/medvezhonokok239/hw5-photos') if IS_KAGGLE else Path('./photos')
CKPT_DIR = Path('/kaggle/working/train_ckpts') if IS_KAGGLE else Path('./train_ckpts')

for d in [WEIGHTS_DIR, RESULTS_DIR, CKPT_DIR]:
    d.mkdir(exist_ok=True)

In [ ]:
IMAGE_SIZE = 512
TRIGGER_TOKEN = 'ohwx'
SUBJECT_CLASS = 'person'
INSTANCE_PROMPT = f'a photo of {TRIGGER_TOKEN} {SUBJECT_CLASS}'

TRAIN_STEPS = 2000
LR = 1e-4
LORA_RANK = 8
BATCH_SIZE = 1

print(f'Instance prompt: "{INSTANCE_PROMPT}"')

In [ ]:
class PersonPhotoDataset(Dataset):
    def __init__(self, photos_dir: Path, size: int = 512):
        exts = {'.jpg', '.jpeg', '.png'}
        self.paths = sorted([
            p for p in photos_dir.iterdir()
            if p.suffix.lower() in exts
        ])
        self.transform = transforms.Compose([
            transforms.Resize(size, interpolation=transforms.InterpolationMode.BILINEAR),
            transforms.CenterCrop(size),
            transforms.ToTensor(),
            transforms.Normalize([0.5], [0.5]),
        ])

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        return self.transform(Image.open(self.paths[idx]).convert('RGB'))


photo_ds = PersonPhotoDataset(PHOTOS_DIR, size=IMAGE_SIZE)
print(f'Photos found: {len(photo_ds)}')

## Загрузка и подготовка модели

In [ ]:
tokenizer = CLIPTokenizer.from_pretrained(SD_MODEL, subfolder='tokenizer')
text_encoder = CLIPTextModel.from_pretrained(SD_MODEL, subfolder='text_encoder', torch_dtype=torch.float16)
vae = AutoencoderKL.from_pretrained(SD_MODEL, subfolder='vae', torch_dtype=torch.float16)
unet = UNet2DConditionModel.from_pretrained(SD_MODEL, subfolder='unet', torch_dtype=torch.float32)
noise_sched = DDPMScheduler.from_pretrained(SD_MODEL, subfolder='scheduler')

vae.requires_grad_(False)
text_encoder.requires_grad_(False)
unet.requires_grad_(False)

lora_cfg = LoraConfig(
    r=LORA_RANK,
    lora_alpha=LORA_RANK,
    init_lora_weights='gaussian',
    target_modules=['to_q', 'to_k', 'to_v', 'to_out.0'],
)
unet.add_adapter(lora_cfg)

trainable = sum(p.numel() for p in unet.parameters() if p.requires_grad)
total = sum(p.numel() for p in unet.parameters())
print(f'LoRA params: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)')

## Предвычисление латентов и текстовых эмбеддингов

In [ ]:
vae.to(DEVICE).eval()
cached_latents = []

with torch.no_grad():
    for i in range(len(photo_ds)):
        pixel = photo_ds[i].unsqueeze(0).to(DEVICE, dtype=torch.float16)
        lat = vae.encode(pixel).latent_dist.sample() * vae.config.scaling_factor
        cached_latents.append(lat.float().cpu())

all_latents = torch.cat(cached_latents)
print(f'Latents shape: {all_latents.shape}')

del vae
torch.cuda.empty_cache() if DEVICE == 'cuda' else (torch.cuda.empty_cache() if DEVICE == 'cuda' else (torch.mps.empty_cache() if DEVICE == 'mps' else None))
gc.collect()

In [ ]:
text_encoder.to(DEVICE).eval()

with torch.no_grad():
    tok_out = tokenizer(
        INSTANCE_PROMPT,
        padding='max_length',
        max_length=tokenizer.model_max_length,
        truncation=True,
        return_tensors='pt',
    )
    text_emb = text_encoder(tok_out.input_ids.to(DEVICE))[0].float().cpu()

print(f'Text embedding shape: {text_emb.shape}')

del text_encoder
torch.cuda.empty_cache() if DEVICE == 'cuda' else (torch.cuda.empty_cache() if DEVICE == 'cuda' else (torch.mps.empty_cache() if DEVICE == 'mps' else None))
gc.collect()

## Обучение LoRA

In [ ]:
unet.to(DEVICE)
unet.enable_gradient_checkpointing()

optimizer = torch.optim.AdamW(
    [p for p in unet.parameters() if p.requires_grad],
    lr=LR,
    weight_decay=1e-2,
)

cond_emb = text_emb.to(DEVICE)

cur_step = 0
loss_log = []
ckpt_file = CKPT_DIR / 'latest.pt'

if ckpt_file.exists():
    saved = torch.load(ckpt_file, map_location='cpu', weights_only=False)
    unet.load_state_dict(saved['unet'], strict=False)
    optimizer.load_state_dict(saved['optimizer'])
    for st in optimizer.state.values():
        for k, v in st.items():
            if isinstance(v, torch.Tensor):
                st[k] = v.to(DEVICE)
    cur_step = saved['step'] + 1
    loss_log = saved['loss_log']
    print(f'Resumed from step {cur_step}, last loss={loss_log[-1]:.4f}')
else:
    print('Starting fresh.')

In [ ]:
unet.train()
pbar = tqdm(range(cur_step, TRAIN_STEPS), initial=cur_step, total=TRAIN_STEPS)

for step in pbar:
    idx = torch.randint(0, len(all_latents), (BATCH_SIZE,))
    latents = all_latents[idx].to(DEVICE)

    noise = torch.randn_like(latents)
    timesteps = torch.randint(
        0, noise_sched.config.num_train_timesteps, (BATCH_SIZE,), device=DEVICE
    ).long()

    noisy = noise_sched.add_noise(latents, noise, timesteps)
    pred = unet(noisy, timesteps, cond_emb, return_dict=False)[0]

    loss = F.mse_loss(pred.float(), noise.float())
    loss.backward()
    torch.nn.utils.clip_grad_norm_(unet.parameters(), 1.0)
    optimizer.step()
    optimizer.zero_grad()

    loss_log.append(loss.item())
    if step % 10 == 0:
        pbar.set_postfix(loss=f'{loss.item():.4f}')

    if (step + 1) % 50 == 0 or step == TRAIN_STEPS - 1:
        torch.save({
            'step': step,
            'unet': {k: v.cpu() for k, v in unet.state_dict().items() if 'lora' in k},
            'optimizer': optimizer.state_dict(),
            'loss_log': loss_log,
        }, ckpt_file)

del optimizer, cond_emb
torch.cuda.empty_cache() if DEVICE == 'cuda' else (torch.cuda.empty_cache() if DEVICE == 'cuda' else (torch.mps.empty_cache() if DEVICE == 'mps' else None))
gc.collect()

plt.figure(figsize=(10, 4))
plt.plot(loss_log, alpha=0.3, label='loss')
if len(loss_log) >= 50:
    plt.plot(np.convolve(loss_log, np.ones(50)/50, mode='valid'), label='moving avg (50)')
plt.xlabel('Step'); plt.ylabel('MSE Loss')
plt.title('LoRA Training Loss')
plt.legend(); plt.grid(True, alpha=0.3)
plt.show()
print(f'Final avg loss: {np.mean(loss_log[-50:]):.4f}')

In [ ]:
unet.save_pretrained(str(WEIGHTS_DIR))
print(f'LoRA weights saved to {WEIGHTS_DIR}')

## Генерация изображений

In [ ]:
unet.eval()
for v in ['all_latents', 'text_emb']:
    if v in dir():
        exec(f'del {v}')
torch.cuda.empty_cache() if DEVICE == 'cuda' else (torch.cuda.empty_cache() if DEVICE == 'cuda' else (torch.mps.empty_cache() if DEVICE == 'mps' else None))
gc.collect()

pipe = StableDiffusionPipeline.from_pretrained(
    SD_MODEL,
    unet=unet,
    torch_dtype=torch.float32,
    safety_checker=None,
    requires_safety_checker=False,
).to(DEVICE)
pipe.enable_attention_slicing()


def generate_image(prompt, seed=SEED, n_steps=30, guidance=7.5):
    gen = torch.Generator(device='cpu').manual_seed(seed)
    return pipe(
        prompt,
        num_inference_steps=n_steps,
        guidance_scale=guidance,
        generator=gen,
    ).images[0]

### 4.1 Творческие промпты

In [ ]:
creative_prompts = [
    f'a photo of {TRIGGER_TOKEN} {SUBJECT_CLASS} in a cyberpunk city, neon lights, futuristic, detailed',
    f'a photo of {TRIGGER_TOKEN} {SUBJECT_CLASS} made of shiny chrome metal, studio lighting',
    f'a photo of {TRIGGER_TOKEN} {SUBJECT_CLASS} in an enchanted forest, magical glow, fantasy art',
    f'a photo of {TRIGGER_TOKEN} {SUBJECT_CLASS} as a medieval knight, castle background, cinematic',
    f'a photo of {TRIGGER_TOKEN} {SUBJECT_CLASS} in a spacesuit on Mars, red planet, sci-fi atmosphere',
]

creative_imgs = []
fig, axes = plt.subplots(1, 5, figsize=(25, 5))

for i, (prompt, ax) in enumerate(zip(creative_prompts, axes)):
    img = generate_image(prompt, seed=SEED + i)
    creative_imgs.append(img)
    short = prompt.split(',')[0].replace(f'a photo of {TRIGGER_TOKEN} {SUBJECT_CLASS} ', '')
    ax.imshow(img)
    ax.set_title(short, fontsize=8)
    ax.axis('off')
    img.save(RESULTS_DIR / f'creative_{i+1}.png')

plt.tight_layout()
plt.show()

### 4.2 Обязательные промпты (forest, city, beach)

In [ ]:
class_word = SUBJECT_CLASS

required_prompts = [
    f'ohwx {class_word} in a forest, high quality, realism',
    f'{class_word} in a forest, high quality, realism',
    f'ohwx {class_word} in a city, high quality, realism',
    f'{class_word} in a city, high quality, realism',
    f'ohwx {class_word} on a beach, high quality, realism',
    f'{class_word} on a beach, high quality, realism',
]
req_labels = ['ohwx_Forest', 'Forest', 'ohwx_City', 'City', 'ohwx_Beach', 'Beach']

required_imgs = []
fig, axes = plt.subplots(1, 6, figsize=(24, 5))

for i, (prompt, label, ax) in enumerate(zip(required_prompts, req_labels, axes)):
    img = generate_image(prompt, seed=SEED)
    required_imgs.append(img)
    ax.imshow(img)
    ax.set_title(f'{label}\n"{prompt}"', fontsize=8)
    ax.axis('off')
    img.save(RESULTS_DIR / f'required_{label.lower()}.png')

plt.tight_layout()
plt.show()

In [ ]:
del pipe
torch.cuda.empty_cache() if DEVICE == 'cuda' else (torch.cuda.empty_cache() if DEVICE == 'cuda' else (torch.mps.empty_cache() if DEVICE == 'mps' else None))
gc.collect()

## Метрики CLIP-T, CLIP-I, Sharpness

In [ ]:
from scipy.signal import convolve2d

clip_model = CLIPModel.from_pretrained('openai/clip-vit-base-patch32').to(DEVICE)
clip_proc = CLIPProcessor.from_pretrained('openai/clip-vit-base-patch32')
clip_model.eval()


def clip_text_similarity(image, text):
    inputs = clip_proc(text=[text], images=image, return_tensors='pt', padding=True)
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
    with torch.no_grad():
        out = clip_model(**inputs)
    return (out.logits_per_image / 100).item()


def clip_image_embedding(image):
    inp = clip_proc(images=image, return_tensors='pt')
    pv = inp['pixel_values'].to(DEVICE)
    with torch.no_grad():
        feat = clip_model.vision_model(pixel_values=pv).pooler_output
        emb = clip_model.visual_projection(feat)
    return emb / emb.norm(dim=-1, keepdim=True)


def clip_identity_similarity(gen_img, ref_imgs):
    gen_emb = clip_image_embedding(gen_img)
    sims = [F.cosine_similarity(gen_emb, clip_image_embedding(ref)).item() for ref in ref_imgs]
    return float(np.mean(sims))


def laplacian_sharpness(image):
    gray = np.array(image.convert('L'), dtype=np.float64)
    lap = np.array([[0, 1, 0], [1, -4, 1], [0, 1, 0]], dtype=np.float64)
    return convolve2d(gray, lap, mode='same').var()


ref_photos = [
    Image.open(p).convert('RGB')
    for p in sorted(PHOTOS_DIR.iterdir())
    if p.suffix.lower() in {'.jpg', '.jpeg', '.png'}
]

In [ ]:
all_gen = creative_imgs + required_imgs
all_prompts = creative_prompts + required_prompts
all_labels = [
    'Cyberpunk', 'Metal', 'Enchanted Forest', 'Medieval', 'Mars',
    'Forest (ohwx)', 'Forest', 'City (ohwx)', 'City', 'Beach (ohwx)', 'Beach'
]

rows = []
for img, prompt, label in zip(all_gen, all_prompts, all_labels):
    ct = clip_text_similarity(img, prompt)
    ci = clip_identity_similarity(img, ref_photos)
    sharp = laplacian_sharpness(img)
    rows.append({'Scene': label, 'CLIP-T': ct, 'CLIP-I': ci, 'Sharpness': sharp})

results_df = pd.DataFrame(rows)

print('=' * 65)
print('МЕТРИКИ КАЧЕСТВА ГЕНЕРАЦИИ')
print('=' * 65)
print(results_df.to_string(index=False, float_format='%.4f'))
print('-' * 65)
print(f'Среднее CLIP-T:    {results_df["CLIP-T"].mean():.4f}')
print(f'Среднее CLIP-I:    {results_df["CLIP-I"].mean():.4f}')
print(f'Среднее Sharpness: {results_df["Sharpness"].mean():.1f}')